In [1]:
import pandas as pd
import numpy as np
import random
import os
import uuid
from datetime import datetime, timedelta

# parametreler ve isim şirket randomları
first_names = ["Ahmet", "Mehmet", "Mustafa", "Ali", "Hüseyin", "Hasan", "İbrahim", "Yusuf", "Ayşe", "Fatma", "Emine", "Hatice", "Zeynep", "Elif", "Burak", "Mert", "Selin", "Deniz", "Gizem", "Buse"]
last_names = ["Yılmaz", "Kaya", "Demir", "Çelik", "Şahin", "Yıldız", "Yıldırım", "Öztürk", "Aydın", "Özdemir", "Arslan", "Doğan", "Kılıç", "Aslan", "Çetin", "Kara", "Koç", "Kurt", "Özkan", "Şimşek"]
company_keywords = ["Tekstil", "İnşaat", "Lojistik", "Gıda", "Sanayi", "Ticaret", "Enerji", "Metal", "Otomotiv", "Turizm", "Plastik", "Kimya", "Yazılım"]
legal_forms = ["Ltd. Şti.", "A.Ş.", "Şahıs Firması"]

def generate_dynamic_name():
    if random.random() < 0.65:
        name = random.choice(last_names)
        s1, s2 = random.sample(company_keywords, 2)
        return f"{name} {s1} ve {s2} {random.choice(legal_forms)}"
    return f"{random.choice(first_names)} {random.choice(last_names)}"



In [2]:
#sektör risk bölümü
sector_logic = {
    'Tekstil': {'risk': 0.5, 'tenor': 60, 'tax': 'TEVKİFAT'},
    'İnşaat': {'risk': 0.9, 'tenor': 120, 'tax': 'TEVKİFAT'},
    'Lojistik': {'risk': 0.4, 'tenor': 45, 'tax': 'TEVKİFAT'},
    'Gıda': {'risk': 0.2, 'tenor': 20, 'tax': 'STANDART'},
    'Otomotiv': {'risk': 0.3, 'tenor': 90, 'tax': 'STANDART'},
    'Enerji': {'risk': 0.8, 'tenor': 150, 'tax': 'STANDART'},
    'Sağlık': {'risk': 0.3, 'tenor': 75, 'tax': 'STANDART'},
    'Perakende': {'risk': 0.35, 'tenor': 30, 'tax': 'STANDART'},
    'Metal': {'risk': 0.6, 'tenor': 60, 'tax': 'TEVKİFAT'}
}

In [3]:
#master dataset generation kısmı
rows = []
base_date = datetime(2025, 1, 1)

for i in range(10000):
    sector = random.choice(list(sector_logic.keys()))
    logic = sector_logic[sector]
    
    cust_id = f"CUST-{random.randint(1000, 9999)}"
    cust_name = generate_dynamic_name()
    age = random.randint(22, 65)
    income = random.randint(45000, 5000000)
    inv_id = f"GIB-{str(uuid.uuid4())[:14].upper()}"
    
    invoice_date = base_date + timedelta(days=random.randint(0, 365))
    due_date = invoice_date + timedelta(days=logic['tenor'])
    
    amount = random.randint(15000, 1500000)
    factoring_type = random.choice(['With Recourse', 'Non-Recourse'])
    dpd = int(np.random.gamma(shape=2.5, scale=logic['risk'] * 12))
    
    ptp_prob = 0.85 - (dpd * 0.015) - (logic['risk'] * 0.25)
    if factoring_type == 'Non-Recourse': ptp_prob -= 0.1
    ptp_status = 1 if random.random() < max(0.05, ptp_prob) else 0
    contact_type = 'CALL' if dpd > 5 else random.choice(['SMS', 'CALL'])

    rows.append({
        'customer_id': cust_id,
        'customer_name': cust_name,
        'age': age,
        'income': income,
        'sector': sector,
        'legal_type': 'Corporate (VKN)' if any(x in cust_name for x in ["A.Ş.", "Ltd.", "Şti."]) else 'Individual (TCKN)',
        'invoice_id': inv_id,
        'invoice_date': invoice_date.strftime('%Y-%m-%d'),
        'due_date': due_date.strftime('%Y-%m-%d'),
        'invoice_amount': amount,
        'tax_type': logic['tax'],
        'factoring_type': factoring_type,
        'dpd': dpd,
        'contact_type': contact_type,
        'ptp_status': ptp_status
    })

In [4]:
#dosya kayıt bloğu
df = pd.DataFrame(rows)
target_directory = r"C:\Users\yildi\Desktop\tamfinans"
file_name = "collection_master_data_v2.csv"

if not os.path.exists(target_directory):
    os.makedirs(target_directory)

full_path = os.path.join(target_directory, file_name)
df.to_csv(full_path, index=False, encoding='utf-8-sig')

print(f"Dataset v2 başarıyla oluşturuldu: {full_path}")

Dataset v2 başarıyla oluşturuldu: C:\Users\yildi\Desktop\tamfinans\collection_master_data_v2.csv
